In [19]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [24]:
bc=datasets.load_breast_cancer()
df=pd.DataFrame(data=bc.data,columns=bc.feature_names)
x,y=bc.data,bc.target
n_samples,n_features=x.shape
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=100)
sc=StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.fit_transform(x_test)
x_train=torch.from_numpy(x_train.astype(np.float32))
x_test=torch.from_numpy(x_test.astype(np.float32))
y_train=torch.from_numpy(y_train.astype(np.float32))
y_test=torch.from_numpy(y_test.astype(np.float32))
y_train=y_train.view(y_train.shape[0],1)
y_test=y_test.view(y_test.shape[0],1)

class LogisticRegression(nn.Module):
    def __init__(self,n_input_features):
        super(LogisticRegression,self).__init__()
        self.linear=nn.Linear(n_input_features,1)
    
    def forward(self,x):
        y_predicted=torch.sigmoid(self.linear(x))
        return y_predicted
    
model=LogisticRegression(n_features)

# loss and optimizer
learning_rate=0.001
criterion=nn.BCELoss()
optimizer=torch.optim.SGD(model.parameters(),lr=learning_rate)


# training 
num_epochs=10000
for epoch in range(num_epochs):
    # forward pass
    y_predicted=model(x_train)
    # backward pass
    loss=criterion(y_predicted,y_train)
    loss.backward()
    #update parameters
    optimizer.step()
    optimizer.zero_grad()
    if epoch % 1000 == 0:
        print(f"epoch {epoch} loss : {loss.item():.5f}")

with torch.no_grad():
    y_predicted=model(x_test)
    y_predicted_cls=y_predicted.round()
    acc=y_predicted_cls.eq(y_test).sum()/float(y_test.shape[0])
    print(f"accuracy : {acc:.4f}")

epoch 0 loss : 0.68935
epoch 1000 loss : 0.24574
epoch 2000 loss : 0.18474
epoch 3000 loss : 0.15638
epoch 4000 loss : 0.13918
epoch 5000 loss : 0.12743
epoch 6000 loss : 0.11880
epoch 7000 loss : 0.11216
epoch 8000 loss : 0.10686
epoch 9000 loss : 0.10252
accuracy : 0.9561
